# Sentinel AI — System Walkthrough

**Author:** Majed Mohamed Alsehli  
**Training programme:** SDAIA Academy — Building Agentic AI Systems  
**Instructor:** محمد البلادي  
**Cohort:** 23–27 August 2026  
**Declared track:** A — Supervisor + workers

This notebook runs the complete Sentinel AI workflow from environment setup through LangSmith trace inspection.

## 1. Environment setup

Install the project dependencies and verify the configured model and tracing services. Optional reputation-provider keys are shown only as booleans.

In [1]:
%pip install -q --disable-pip-version-check -r ../requirements.txt

import json
import os
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
load_dotenv('../.env', override=True)
sys.path.insert(0, '../src')

from sentinel.config import PROJECT_ROOT, langsmith_status
status = langsmith_status()
credential_status = {
    'OPENAI_API_KEY': bool(os.getenv('OPENAI_API_KEY')),
    'LANGCHAIN_API_KEY': bool(os.getenv('LANGCHAIN_API_KEY') or os.getenv('LANGSMITH_API_KEY')),
    'ABUSEIPDB_API_KEY_optional': bool(os.getenv('ABUSEIPDB_API_KEY')),
    'VIRUSTOTAL_API_KEY_optional': bool(os.getenv('VIRUSTOTAL_API_KEY')),
}
print('Python:', sys.version.split()[0])
print('Credential readiness:', credential_status)
print('LangSmith readiness:', status)
assert credential_status['OPENAI_API_KEY'], 'Configure OPENAI_API_KEY in ../.env'
assert status['tracing_enabled'], 'Set LANGCHAIN_TRACING_V2=true in ../.env'
assert status['api_key_configured'], 'Configure LANGCHAIN_API_KEY in ../.env'
print('Environment ready')

Note: you may need to restart the kernel to use updated packages.


Python: 3.13.2
Credential readiness: {'OPENAI_API_KEY': True, 'LANGCHAIN_API_KEY': True, 'ABUSEIPDB_API_KEY_optional': False, 'VIRUSTOTAL_API_KEY_optional': False}
LangSmith readiness: {'tracing_enabled': True, 'api_key_configured': True, 'project': 'sentinel-ai'}
Environment ready


## 2. Tool execution and structured output

The email specialist receives an argument-dependent parsing tool. The LLM chooses the call, Sentinel executes it, returns a `ToolMessage`, and obtains a Pydantic `SpecialistAssessment`.

In [2]:
from sentinel.agents.specialists import run_email_agent

raw_email = '''From: Account Security <alerts@contoso-security.example>
Reply-To: recovery-team@example.net
Subject: Urgent — verify your account

Your password expires today. Verify your account at https://example.net/login immediately.'''
specialist_demo = run_email_agent(raw_email)
print('Structured result type:', type(specialist_demo).__name__)
print('Specialist assessment:', specialist_demo.assessment.model_dump())
print('Executed tool observations:')
for item in specialist_demo.tool_observations:
    print(json.dumps(item.model_dump(), indent=2, default=str))
assert specialist_demo.tool_observations, 'The model did not choose a tool'
assert specialist_demo.tool_observations[0].tool_name == 'extract_email_indicators'
assert specialist_demo.tool_observations[0].output['urls'] == ['https://example.net/login']
print('Tool execution completed')

Structured result type: SpecialistResult
Specialist assessment: {'summary': 'The email exhibits several characteristics typical of phishing attempts, including urgency, suspicious sender information, and a request for immediate action. It is advisable to verify the legitimacy of the email before clicking on any links or providing personal information.', 'notable_indicators': ['From: Account Security <alerts@contoso-security.example>', 'Reply-To: recovery-team@example.net', 'Subject: Urgent — verify your account', 'URLs: https://example.net/login', "Suspicious Phrases: 'urgent', 'verify your account', 'password expires'"], 'limitations': ["The sender's address may appear legitimate but could be spoofed.", 'The reply-to address is different from the sender, raising suspicion.', 'The analysis does not confirm the actual legitimacy of the URL provided.', 'No attachments were present to analyze for further threats.']}
Executed tool observations:
{
  "tool_name": "extract_email_indicators",


## 3. Multi-agent routing — structured LLM supervisor

Sentinel uses a supervisor because each dominant artifact needs a narrow specialist and tool set. The decision is made by `with_structured_output(RouteDecision)`; Python only dispatches the returned destination.

In [3]:
from sentinel.agents.supervisor import route_request
from sentinel.models.schemas import RouteDecision

routing_examples = [
    'Investigate network traffic from 1.1.1.1',
    'Review this raw email for credential phishing',
    'Check SHA-256 ' + 'a' * 64,
    'Assess https://example.com/login without opening it',
]
route_results = [route_request(question) for question in routing_examples]
for question, decision in zip(routing_examples, route_results):
    print(question, '=>', decision.model_dump())
assert all(isinstance(decision, RouteDecision) for decision in route_results)
assert {decision.destination for decision in route_results} == {
    'ip_agent', 'email_agent', 'file_agent', 'url_agent'
}
print('Structured routing completed')

Investigate network traffic from 1.1.1.1 => {'destination': 'ip_agent', 'reason': 'The investigation focuses on network traffic from a specific IP address (1.1.1.1), making it necessary to engage the IP specialist to analyze the network indicators associated with this address.'}
Review this raw email for credential phishing => {'destination': 'email_agent', 'reason': 'The investigation focuses on a raw email, which includes headers and message body, making it an email artifact. The primary concern is to analyze the content for signs of credential phishing.'}
Check SHA-256 aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa => {'destination': 'file_agent', 'reason': 'The investigation is focused on a SHA-256 hash, which is a file artifact. Therefore, the file specialist is the appropriate choice.'}
Assess https://example.com/login without opening it => {'destination': 'url_agent', 'reason': "The investigation is focused on assessing the URL 'https://example.com/login', whic

## 4. Hybrid RAG — load, split, embed, store, retrieve

Sentinel uses **Hybrid RAG**: the local corpus supplies stable defensive procedures while live tools supply current reputation and infrastructure information.

In [4]:
from sentinel.rag.loader import load_markdown_documents
from sentinel.rag.retriever import build_retriever
from sentinel.workflows.workflow import set_retriever

documents = load_markdown_documents()
retriever, chunks = build_retriever(documents, PROJECT_ROOT / 'chroma')
set_retriever(retriever)
print('Loaded documents:', len(documents))
print('Embedded/stored chunks:', len(chunks))
print('Sources:', [document.metadata['filename'] for document in documents])
assert len(documents) == 5 and len(chunks) >= 5
print('RAG index ready')

Loaded documents: 5
Embedded/stored chunks: 10
Sources: ['incident_response.md', 'ip_reputation.md', 'malware.md', 'phishing.md', 'url_analysis.md']
RAG index ready


In [5]:
rag_question = 'What safe initial response should an analyst take for a phishing message?'
expected_phrase = 'isolate the message, preserve full headers, and avoid opening links or attachments'
rag_hits = retriever.invoke(rag_question)
print('Retrieved chunks:', len(rag_hits))
for number, hit in enumerate(rag_hits, 1):
    print(f'[{number}] {hit.metadata.get("filename", hit.metadata.get("source"))}')
    print(hit.page_content[:500], '\n')
retrieved_text = '\n'.join(hit.page_content for hit in rag_hits).lower()
assert expected_phrase in retrieved_text
print('Retrieval completed')

Retrieved chunks: 4
[1] phishing.md
# Phishing Indicators

Phishing assessment should combine message content, sender identity, link targets, attachments, and authentication results. Common indicators include urgent requests, suspicious sender domains, mismatched links, credential requests, unexpected attachments, and fear or urgency language. No single wording cue proves malicious intent.

Preserve the original message and full headers before taking action. A safe initial response is to isolate the message, preserve full headers, 

[2] phishing.md
Useful header fields include `From`, `Reply-To`, `Return-Path`, received hops, and SPF, DKIM, and DMARC results. A display name that imitates a trusted organization while the underlying address uses an unrelated domain is a strong impersonation signal. Authentication failures matter, but legitimate forwarding can complicate interpretation.

Analysts should document which indicators were directly observed, which were returned by reputation se

## 5. Context, short-term state, and long-term memory

Short-term workflow state uses an `InMemorySaver` and explicit `thread_id`. Long-term facts use a separate `InMemoryStore`. The following run writes in thread A and reads in thread B under the same analyst namespace.

In [6]:
from sentinel.memory.store import cross_thread_memory_workflow

memory_analyst = 'demo-analyst'
thread_a_config = {'configurable': {'thread_id': 'memory-thread-A'}}
thread_b_config = {'configurable': {'thread_id': 'memory-thread-B'}}
memory_write = cross_thread_memory_workflow.invoke({
    'operation': 'write', 'thread_label': 'thread-A', 'analyst_id': memory_analyst,
    'key': 'report_format', 'value': 'PDF'
}, thread_a_config)
memory_read = cross_thread_memory_workflow.invoke({
    'operation': 'read', 'thread_label': 'thread-B', 'analyst_id': memory_analyst,
    'key': 'report_format'
}, thread_b_config)
print('Write result:', memory_write)
print('Read result:', memory_read)
assert thread_a_config['configurable']['thread_id'] != thread_b_config['configurable']['thread_id']
assert memory_read['value'] == 'PDF'
print('Cross-thread memory read completed')

Write result: {'operation': 'write', 'thread_label': 'thread-A', 'analyst_id': 'demo-analyst', 'key': 'report_format', 'value': 'PDF'}
Read result: {'operation': 'read', 'thread_label': 'thread-B', 'analyst_id': 'demo-analyst', 'key': 'report_format', 'value': 'PDF'}
Cross-thread memory read completed


## 6. Human-in-the-loop — interrupt and resume

The full workflow uses `force_human_review=True` to pause before PDF persistence. The first invocation captures the interrupt and checkpoint; the second resumes the same thread with a structured reviewer decision.

In [7]:
from sentinel.workflows.workflow import sentinel_workflow

demo_started_at = datetime.now(timezone.utc)
hitl_config = {'configurable': {'thread_id': 'sentinel-hitl-demo'}}
workflow_input = {
    'request': 'Review this complete raw email for phishing indicators.\n\n' + raw_email,
    'analyst_id': memory_analyst,
    'force_human_review': True,
    'output_path': str(PROJECT_ROOT / 'reports' / 'approved_incident.pdf'),
}
first_run = sentinel_workflow.invoke(workflow_input, hitl_config)
print('First invocation:', first_run)
assert '__interrupt__' in first_run
interrupt_payload = first_run['__interrupt__'][0].value
assert interrupt_payload['action'] == 'persist_final_incident_report'
checkpoint = sentinel_workflow.get_state(hitl_config)
print('Saved checkpoint next tasks:', checkpoint.next)
print('Interrupt captured')

First invocation: {'__interrupt__': [Interrupt(value={'action': 'persist_final_incident_report', 'question': 'Approve persisting this incident report?', 'analysis': {'verdict': 'suspicious', 'confidence': 0.8, 'threat_type': 'phishing', 'explanation': "The email contains several indicators commonly associated with phishing attempts, such as urgency in the subject line and a request for account verification. The sender's email address and reply-to address are also suspicious, as they do not match typical formats from legitimate organizations. However, without verification of the sender's authenticity and the actual status of the account, there is uncertainty regarding the true nature of this email.", 'findings': ['Urgent subject line requesting account verification', 'Suspicious sender and reply-to addresses', 'Link provided in the email could be deceptive'], 'recommendations': ['Isolate the email and preserve full headers', 'Do not click on any links or open attachments', "Verify the s

In [8]:
from langgraph.types import Command

review_decision = {
    'approved': True,
    'reviewer': 'Security analyst',
    'reason': 'The findings and limitations were reviewed before report persistence.',
}
resumed_run = sentinel_workflow.invoke(Command(resume=review_decision), hitl_config)
print(json.dumps(resumed_run, indent=2, default=str))
assert resumed_run['report']['approved'] is True
assert resumed_run['executed_tool_count'] >= 1
assert resumed_run['retrieved_documents'] >= 1
assert resumed_run['finalization']['status'] == 'written'
assert Path(resumed_run['finalization']['output_path']).exists()
print('Resume completed')

{
  "route": {
    "destination": "email_agent",
    "reason": "The investigation involves reviewing a complete raw email for phishing indicators, which falls under the domain of the email_agent."
  },
  "specialist": {
    "destination": "email_agent",
    "assessment": {
      "summary": "The email exhibits several characteristics commonly associated with phishing attempts, including urgency and requests for account verification.",
      "notable_indicators": [
        "From Address: Account Security <alerts@contoso-security.example>",
        "Reply-To Address: recovery-team@example.net",
        "Subject: Urgent \u2014 verify your account",
        "URLs: https://example.net/login",
        "Suspicious Phrases: urgent, verify your account, password expires"
      ],
      "limitations": [
        "The analysis is based solely on the content of the email and does not include verification of the sender's authenticity.",
        "The tool used may not detect all phishing attempts, esp

## 7. Functional API and two error strategies

Sentinel uses LangGraph `@task` and `@entrypoint`. Model-dependent tasks have a bounded `RetryPolicy`, while permanent investigation failures use a controlled `unknown` fallback. The transient exercise succeeds on the third framework-managed attempt, and the permanent failure reaches fallback.

In [9]:
from sentinel.workflows.reliability_demo import reliability_demo
from sentinel.workflows.workflow import external_retry

reliability_result = reliability_demo.invoke({'run_id': str(uuid4())})
print('Production RetryPolicy:', external_retry)
print('Reliability exercise:', reliability_result)
assert reliability_result['retry_result']['attempts'] == 3
assert reliability_result['retry_result']['status'] == 'recovered'
assert reliability_result['fallback_result']['strategy'] == 'controlled_fallback'
assert reliability_result['fallback_result']['verdict'] == 'unknown'
print('Retry recovery completed')
print('Controlled fallback completed')

Production RetryPolicy: RetryPolicy(initial_interval=0.2, backoff_factor=2.0, max_interval=2.0, max_attempts=3, jitter=False, retry_on=<function default_retry_on at 0x1199e23e0>)
Reliability exercise: {'retry_policy': 'RetryPolicy', 'retry_result': {'status': 'recovered', 'attempts': 3}, 'fallback_result': {'verdict': 'unknown', 'confidence': 0.0, 'strategy': 'controlled_fallback', 'handled_error': 'ValueError'}}
Retry recovery completed
Controlled fallback completed


## 8. Workflow pattern — Routing

**Declared workflow pattern: Routing.** It fits because email, URL, IP, and file artifacts are mutually exclusive dominant investigation modes with different tool permissions. An LLM supervisor makes the choice with structured output; shared retrieval, synthesis, approval, reporting, and memory stages then enforce consistent behavior.

In [10]:
print('Workflow Pattern: Routing')
print('Routing authority: structured LLM supervisor')
print('Why it fits: mutually exclusive artifact specialists with narrow tools')
assert resumed_run['route']['destination'] == 'email_agent'
print('End-to-end routing completed')

Workflow Pattern: Routing
Routing authority: structured LLM supervisor
Why it fits: mutually exclusive artifact specialists with narrow tools
End-to-end routing completed


## 9. LangSmith trace observation

The next cell waits for trace upload, queries the configured LangSmith project, finds the current workflow run, and calculates an observation from its timings and child runs.

In [11]:
import warnings

from langchain_core.tracers.langchain import wait_for_all_tracers
from langsmith import Client

warnings.filterwarnings('ignore', category=DeprecationWarning, message=r'list_runs\(\).*')
wait_for_all_tracers()
project_name = str(status['project'])
client = Client()
recent_runs = list(client.list_runs(
    project_name=project_name, start_time=demo_started_at - timedelta(seconds=5), limit=100
))
workflow_runs = [run for run in recent_runs if run.name == 'sentinel_workflow']
assert workflow_runs, f'No Sentinel workflow run found in LangSmith project {project_name}'
# Functional API traces use a LangGraph root with sentinel_workflow as a child.
# The earliest workflow run is the full pre-interrupt investigation; resume is a later trace.
workflow_run = min(workflow_runs, key=lambda run: run.start_time)
import time
for _ in range(8):
    trace_runs = list(client.list_runs(project_name=project_name, trace_id=workflow_run.trace_id))
    if any(run.run_type == 'tool' for run in trace_runs):
        break
    time.sleep(2)
trace_root = next(run for run in trace_runs if run.parent_run_id is None)
completed_runs = [run for run in trace_runs if run.end_time is not None]
assert completed_runs, 'The trace has no completed runs'
duration = lambda run: (run.end_time - run.start_time).total_seconds()
slowest = max(completed_runs, key=duration)
error_count = sum(bool(run.error) for run in trace_runs)
tool_run_count = sum(run.run_type == 'tool' for run in trace_runs)
trace_observation = (
    f'Actual trace observation: trace {trace_root.trace_id} contained {len(trace_runs)} runs; '
    f'the slowest run was {slowest.name} at {duration(slowest):.3f}s; '
    f'{error_count} run(s) recorded errors; and the trace recorded '
    f'{tool_run_count} model-selected tool run(s).'
)
print('LangSmith project:', project_name)
print('Root trace ID:', trace_root.trace_id)
print(trace_observation)
assert len(trace_runs) > 1 and tool_run_count >= 1
print('LangSmith trace inspected')

LangSmith project: sentinel-ai
Root trace ID: 01a03d23-877c-7a20-bd3a-3504fe5e2e26
Actual trace observation: trace 01a03d23-877c-7a20-bd3a-3504fe5e2e26 contained 18 runs; the slowest run was specialist_investigation at 6.251s; 0 run(s) recorded errors; and the trace recorded 1 model-selected tool run(s).
LangSmith trace inspected


## 10. Automated regression suite

The suite covers tool execution, schemas, structured routing, deterministic RAG retrieval, cross-thread memory, HITL resume, both error strategies, and PDF generation.

In [12]:
import subprocess

test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=PROJECT_ROOT, text=True, capture_output=True, check=False
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0
print('Regression tests passed')

...............                                                          [100%]
15 passed in 1.71s

Regression tests passed


## 11. Run summary

Collect the main results from the completed workflow run.

In [13]:
run_summary = {
    'model_selected_tools': len(specialist_demo.tool_observations),
    'routing_destinations': sorted(decision.destination for decision in route_results),
    'rag_chunks': len(chunks),
    'cross_thread_memory': memory_read['value'],
    'hitl_resumed_and_approved': resumed_run['report']['approved'],
    'pdf_written': resumed_run['finalization']['status'],
    'retry_attempts': reliability_result['retry_result']['attempts'],
    'workflow_pattern': 'Routing',
    'langsmith_trace_id': str(trace_root.trace_id),
    'tests_passed': test_run.returncode == 0,
}
print(json.dumps(run_summary, indent=2))
assert all([
    run_summary['model_selected_tools'] >= 1,
    run_summary['cross_thread_memory'] == 'PDF',
    run_summary['hitl_resumed_and_approved'],
    run_summary['pdf_written'] == 'written',
    run_summary['retry_attempts'] == 3,
    run_summary['tests_passed'],
])
print('SYSTEM WALKTHROUGH COMPLETED')

{
  "model_selected_tools": 1,
  "routing_destinations": [
    "email_agent",
    "file_agent",
    "ip_agent",
    "url_agent"
  ],
  "rag_chunks": 10,
  "cross_thread_memory": "PDF",
  "hitl_resumed_and_approved": true,
  "pdf_written": "written",
  "retry_attempts": 3,
  "workflow_pattern": "Routing",
  "langsmith_trace_id": "01a03d23-877c-7a20-bd3a-3504fe5e2e26",
  "tests_passed": true
}
SYSTEM WALKTHROUGH COMPLETED
